# DENTEX — abnormal-tooth detection on panoramic x-rays

Two Mask2Former models: one numbers every tooth (FDI 11–48), one finds the
pathology and says which tooth it sits on. Output = boxes + masks + a verdict
like *"Deep Caries on tooth 47"*.

Run the cells in order. On a Colab **T4** the whole notebook is roughly:

| step | time |
|---|---|
| install detectron2 + Mask2Former | ~10 min |
| download DENTEX (streamed, ~5 GB) | ~15 min |
| stage 2 — tooth detection | ~1.5 h |
| stage 3 — pathology | ~1.5 h |
| evaluation + figures | ~5 min |

**Runtime → Change runtime type → GPU** before you start.

## 0. Environment

In [ ]:
!nvidia-smi
import torch, sys
print("python", sys.version.split()[0])
print("torch ", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

## 1. Install detectron2 and Mask2Former

Two things go wrong most often here, so both are guarded against:

* **Do not reinstall torch.** detectron2 compiles against whatever torch Colab
  already has; replacing it afterwards breaks the compiled extension.
* The `ops` build compiles the MSDeformAttn CUDA kernel. If it fails,
  Mask2Former falls back to a pure-PyTorch path — slower, but training still
  runs, so a failure here is not fatal.

In [ ]:
%%capture install_log
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git' --no-build-isolation
!pip install -q huggingface_hub pycocotools opencv-python-headless
![ -d /content/Mask2Former ] || git clone -q https://github.com/facebookresearch/Mask2Former.git /content/Mask2Former
%cd /content/Mask2Former/mask2former/modeling/pixel_decoder/ops
!python setup.py build install
%cd /content

In [ ]:
# Check the install; print the tail of the log only if something is missing.
import sys
sys.path.insert(0, "/content/Mask2Former")

ok = True
try:
    import detectron2
    print("detectron2", detectron2.__version__)
except Exception as e:
    ok = False; print("detectron2 FAILED:", e)
try:
    from mask2former import add_maskformer2_config
    print("mask2former   ok")
except Exception as e:
    ok = False; print("mask2former FAILED:", e)
try:
    import MultiScaleDeformableAttention
    print("MSDeformAttn  ok (compiled CUDA kernel)")
except Exception as e:
    print("MSDeformAttn  not compiled ->", e)
    print("              falling back to the pure-PyTorch path (slower, still correct)")

if not ok:
    install_log.show()

## 2. The project code

Point `PROJECT_DIR` at this repository. Either mount Drive and use the copy
there, or upload the `Project/` folder to `/content`.

In [ ]:
import os, sys
from pathlib import Path

# --- option A: Google Drive ------------------------------------------------
# from google.colab import drive; drive.mount("/content/drive")
# PROJECT_DIR = Path("/content/drive/MyDrive/ADL/Project")

# --- option B: uploaded to /content ---------------------------------------
PROJECT_DIR = Path("/content/Project")

assert PROJECT_DIR.is_dir(), f"{PROJECT_DIR} not found -- upload the project or mount Drive"
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

import dentex
from dentex import fdi
print("project", PROJECT_DIR, "| dentex", dentex.__version__)
print("example:", fdi.tooth_long_name(36))

In [ ]:
# Logic tests -- no GPU, no dataset. Catches path and version problems early.
!python tests/test_pipeline_logic.py

## 3. Download DENTEX

Streaming mode reads the remote zip's index and extracts only the members the
supervised pipeline needs, so the 1571-image unlabelled split is skipped:
~5 GB instead of 10.9 GB.

The 50-image official validation release is downloaded separately and kept as
an untouched hold-out.

In [ ]:
!python scripts/prepare_data.py --root dataset/dentex --holdout-root dataset/dentex_val

## 4. Look at the data before training on it

Ground truth drawn with the same code that will draw the predictions, so a
mistake in the FDI mapping is visible immediately. Check the orientation: the
`R` marker is on the **left** of a panoramic x-ray, so quadrants 1 and 4
(patient's right) appear on the left-hand side of the image.

In [ ]:
from dentex.data import convert, download
from dentex.inference import contact_sheet, draw_report, polygons_to_mask, report_from_record

gt_json, images_dir = download.find_validation("dataset/dentex_val")
records = convert.load_dentex(gt_json, images_dir, task="disease")
stats = convert.dataset_statistics(records)
print({k: v for k, v in stats.items() if k not in ("teeth", "positions")})

renders = []
for record in [r for r in records if len(r["annotations"]) >= 3][:4]:
    report = report_from_record(record)
    for finding, annotation in zip(report.problems, record["annotations"]):
        finding.mask = polygons_to_mask(annotation["segmentation"], record["width"], record["height"])
    renders.append(draw_report(record["file_name"], report, draw_teeth=False,
                               title=Path(record["file_name"]).name))
    print(report.to_text(), "\n")

contact_sheet(renders, columns=2, thumb_width=900)

## 5. Stage 2 — tooth detection with FDI enumeration

Trained on `quadrant_enumeration`: 634 x-rays with **every** tooth annotated
(28.5 per image), 32 classes laid out as `(quadrant-1)*8 + (position-1)`.

The dry run builds the model and one batch and stops — do it first, it catches
config and dataloader errors in 30 seconds instead of 20 minutes.

In [ ]:
!python scripts/train.py --config configs/dentex_tooth_R50.yaml --dry-run

In [ ]:
# ~1.5 h on a T4. Resume after a disconnect by adding --resume.
!python scripts/train.py --config configs/dentex_tooth_R50.yaml

## 6. Stage 3 — hierarchical pathology

Trained on `quadrant-enumeration-disease`: 705 x-rays where **only abnormal
teeth** are annotated (5.0 per image).

* primary head: diagnosis (4 classes)
* auxiliary heads: quadrant (4) + position (8) → the FDI number of the
  affected tooth

`--init-from` strips the 32-class head from the stage-2 checkpoint and
transfers everything else.

In [ ]:
!python scripts/train.py --config configs/dentex_disease_R50_hier.yaml \
    --init-from outputs/stage2_tooth/model_final.pth

## 7. Evaluate on the untouched hold-out

`triple/recall` — quadrant **and** position **and** diagnosis all correct, as a
share of all ground-truth findings — is the DENTEX criterion and the number to
put in the report.

In [ ]:
!python scripts/train.py --config configs/dentex_disease_R50_hier.yaml --eval-only \
    MODEL.WEIGHTS outputs/stage3_disease/model_final.pth \
    DATASETS.TEST '("dentex_disease_holdout",)'

In [ ]:
# Tooth model on its own internal split (the hold-out only labels abnormal teeth).
!python scripts/train.py --config configs/dentex_tooth_R50.yaml --eval-only \
    MODEL.WEIGHTS outputs/stage2_tooth/model_final.pth

## 8. Run the full pipeline

Both models on the same x-ray, then the findings are bound to teeth by IoU.
When the pathology model's own enumeration heads disagree with the overlapping
tooth box, the tooth detector wins — it has seen 18k annotated teeth against
the pathology model's 3.5k.

In [ ]:
import dentex.modeling  # registers the architecture
from dentex.config import setup_cfg
from dentex.data.register import register_dentex
from dentex.inference import DentexPipeline, contact_sheet, draw_report

register_dentex(root="dataset/dentex", holdout_root="dataset/dentex_val")

tooth_cfg = setup_cfg("configs/dentex_tooth_R50.yaml",
                      ["MODEL.WEIGHTS", "outputs/stage2_tooth/model_final.pth"])
disease_cfg = setup_cfg("configs/dentex_disease_R50_hier.yaml",
                        ["MODEL.WEIGHTS", "outputs/stage3_disease/model_final.pth"])

pipeline = DentexPipeline.from_configs(
    tooth_cfg=tooth_cfg,
    disease_cfg=disease_cfg,
    tooth_score_threshold=0.5,
    disease_score_threshold=0.5,
    link_iou_threshold=0.3,
)
print("pipeline ready")

In [ ]:
from pathlib import Path

xrays = sorted(p for p in Path("dataset/dentex_val").rglob("*.png")
               if ".ipynb_checkpoints" not in p.parts)[:4]

renders = []
for path in xrays:
    report = pipeline(path)
    print(f"== {path.name}")
    print(report.to_text(), "\n")
    renders.append(draw_report(path, report, draw_teeth=True, title=path.name))

contact_sheet(renders, columns=2, thumb_width=900)

In [ ]:
# Predictions next to ground truth, and the markdown table for the write-up.
from dentex.inference import report_from_record, side_by_side

record = records[0]
truth = draw_report(record["file_name"], report_from_record(record),
                    draw_teeth=False, title="ground truth")
predicted = draw_report(record["file_name"], pipeline(record["file_name"]),
                        draw_teeth=True, title="prediction")
display(side_by_side(truth, predicted).resize((1200, 1200 * (truth.height + predicted.height + 16) // truth.width)))

print(pipeline(record["file_name"]).to_markdown())

In [ ]:
# Whole hold-out to disk: annotated pngs + report.json
!python scripts/predict.py \
    --tooth-config configs/dentex_tooth_R50.yaml \
    --tooth-weights outputs/stage2_tooth/model_final.pth \
    --disease-config configs/dentex_disease_R50_hier.yaml \
    --disease-weights outputs/stage3_disease/model_final.pth \
    --images dataset/dentex_val --out outputs/predictions --contact-sheet

## 9. Keep the results

Checkpoints are ~250 MB each. Copy them to Drive before the runtime recycles.

In [ ]:
# from google.colab import drive; drive.mount("/content/drive")
# !mkdir -p /content/drive/MyDrive/ADL/outputs
# !cp outputs/stage2_tooth/model_final.pth  /content/drive/MyDrive/ADL/outputs/tooth.pth
# !cp outputs/stage3_disease/model_final.pth /content/drive/MyDrive/ADL/outputs/disease.pth
# !cp -r outputs/predictions                 /content/drive/MyDrive/ADL/outputs/